# Deployment: recalibration + domain-shift monitoring (UX)

Freeze a **user-chosen** model, then recalibrate progressively drifted in-vivo
data and monitor CORAL domain distance, the confidence signal that flags when the
model is extrapolating beyond its validated regime.

---

**Where this fits: the ElectroPyCal analysis pipeline** (`pip install electropycal`):

1. `raw_spectra_review`: inspect raw EIS/FSCV spectra; choose the analysis band & QC stringency
2. `quality_filtering_dashboard`: set QC gates, see which `(channel, timepoint)` sensors pass, save the shared config
3. `stabilization_review`: confirm each channel's FSCV response stabilized (optional / parallel)
4. `diagnostics_review`: characterize the featureset before modeling (*is recalibration well-posed?*)
5. `discovery_checkpointed`: run the model-discovery task queue batch-by-batch under nested CV
6. `discovery_results_review`: inspect a finished discovery run; pick a model
7. **`deployment_domain_shift`: freeze the chosen model; recalibrate in-vivo data; monitor CORAL domain shift  ← you are here**

Consistency knobs (band, QC gates) are shared across notebooks via `<ROOT>/electropycal_analysis_config.json`. See `docs/USAGE.md`.


## Setup: run first

Installs `electropycal` from PyPI. On Colab it also mounts Google Drive, so you can point
`ROOT` at raw data in your own Drive; skip the mount if your data is local.

```
pip install electropycal
```

After installing on Colab, do **Runtime → Restart session** so the new module loads.


In [ ]:
import sys

if "google.colab" in sys.modules:
    # --upgrade is load-bearing. `pip install electropycal` is a NO-OP when any version is
    # already present, and a hosted runtime often carries an older one from an earlier
    # session, so without it the cell reports success and leaves the old version in place.
    !pip install -q --upgrade electropycal
    from google.colab import drive
    drive.mount("/content/drive")     # only needed if ROOT points into your Drive

# Check the version here, where the message is readable, instead of letting an older build
# fail deep inside a reader. If this raises immediately after an upgrade, restart the
# runtime: a module already imported stays in memory and the upgrade does not reach it.
import electropycal

_parts = electropycal.__version__.split(".")[:2]
if tuple(int(p) for p in _parts if p.isdigit()) < (0, 10):
    raise RuntimeError(
        f"electropycal {electropycal.__version__} is older than these notebooks need "
        f"(0.10 or newer, for the full range of export variants). Run "
        f"`pip install -q --upgrade electropycal`, then restart the runtime so the "
        f"upgraded version is the one imported.")

In [ ]:
%matplotlib inline
from electropycal.viz import set_pub_style
set_pub_style()
import tempfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from electropycal.data.synthetic import make_dataset, make_invivo_drift
from electropycal.selection.cars import cars_select
from electropycal.deployment.deploy import freeze_model, load_frozen_model, recalibrate
from electropycal.deployment.plots import plot_domain_monitor, plot_invivo_recalibration

ds = make_dataset(random_state=0)
# user chooses the model: CARS-selected featureset + linear PLSR, k=3
subset = cars_select(ds.X, ds.y, k_max=3, n_generations=15, random_state=0)
bundle = freeze_model(ds.X, ds.y, ds.feature_names, 'linear_plsr', k=3,
                      out_dir=tempfile.mkdtemp() + '/frozen_model', feature_index=subset)
fm = load_frozen_model(bundle)
print('frozen model settings:')
print(f"  architecture = {fm.manifest['architecture']}  |  k = {fm.manifest['k']}  |  "
      f"features = {fm.manifest['n_features']}")
print("  customize: freeze_model(X, y, names, architecture='weighted_plsr', k=2, feature_index=subset)")

## Recalibrate drifted in-vivo timepoints + monitor domain distance

In [3]:
drift = make_invivo_drift(ds, timepoints=(0, 7, 21, 42))
FLAG = 2.0   # domain-distance threshold beyond which we flag 'extrapolating'
rows = []
for t in (0, 7, 21, 42):
    r = recalibrate(fm, drift[t])
    rows.append(dict(day=t, domain_distance=r['domain_distance'],
                     mean_recal=float(r['norm_ipeak'].mean()),
                     confidence='ok' if r['domain_distance'] < FLAG else 'EXTRAPOLATING'))
df = pd.DataFrame(rows); df

,day,domain_distance,mean_recal,confidence
0,0,0.000004,1.765909,ok
1,7,0.418184,1.770371,ok
2,21,1.262045,1.773076,ok
3,42,2.504306,1.794174,EXTRAPOLATING


In [ ]:
plot_domain_monitor(df, flag=FLAG)

## Recalibrate from a raw in-vivo directory
**Set the two config variables at the top of the next cell and run:**
- `ROOT`: your raw in-vivo directory (e.g. `"data/invivo"`), using the convention
  `<deviceid>_paired_{baseline,live}.csv`.
- `BUNDLE`: a `frozen_model/` directory from `freeze_model` (produced after a real
  in-vitro discovery run).

Leave either as `None` to fall back to a self-contained synthetic demo. The pipeline is
`extract_invivo` → per-sensor D0-normalization vs the early in-vivo baseline → frozen
model → CORAL domain distance per session.

In [ ]:
# === CONFIG - set these two for real data, or leave None for the synthetic demo ===
ROOT   = None     # e.g. "data/invivo"        (raw in-vivo directory: <deviceid>_paired_{baseline,live}.csv)
BUNDLE = None     # e.g. "outputs/frozen_model" (a frozen_model/ dir from a real discovery run)
# ==================================================================================
import tempfile
from electropycal.data.synthetic import write_synthetic_invivo_dir
from electropycal.features.extract import extract_invivo
from electropycal.features.normalize import d0_normalize_frame
from electropycal.deployment.deploy import recalibrate_invivo

if BUNDLE is None:     # demo: freeze a throwaway model on a synthetic in-vivo tree
    tr = extract_invivo(write_synthetic_invivo_dir(tempfile.mkdtemp()+'/train',
                        devices=('3-2',), channels=(3,5,6), sessions=(0,7,21,42)))
    fc = [c for c in tr.columns if c not in ('device','channel','timepoint','time_index','NormIpeak')]
    trn = d0_normalize_frame(tr, fc)
    Xtr = trn[fc].to_numpy(float); ytr = trn['NormIpeak'].to_numpy(float)
    keep = np.isfinite(Xtr).all(0) & (np.nanstd(Xtr,0)>1e-9); fc=[f for f,k in zip(fc,keep) if k]; Xtr=Xtr[:,keep]
    rows = np.isfinite(Xtr).all(1) & np.isfinite(ytr)
    BUNDLE = freeze_model(Xtr[rows], ytr[rows], fc, 'linear_plsr', k=2, out_dir=tempfile.mkdtemp()+'/frozen_model')
# Prefer the shipped demo/in_vivo tree when one is reachable (from ANY cwd);
# synthesize an equivalent only when there is no checkout at all, e.g. Colab.
# ROOT must be a LOCAL FILESYSTEM PATH. A Drive or cloud share URL will not work:
# nothing here downloads for you. On Colab, mount the drive first and use the
# mounted path:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     ROOT = '/content/drive/MyDrive/<your-folder>'
# Point ROOT at the directory that DIRECTLY CONTAINS the <YYYYMMDD>_..._signal
# folders, not at its parent. If you are one level off, extraction now says so and
# names the directory it thinks you meant.
from electropycal._demo import demo_input
ROOT = demo_input('in_vivo', ROOT, devices=('4-2',), channels=(3,5,6), sessions=(0,7,21))
print(f"BUNDLE = {BUNDLE}\nROOT   = {ROOT}  (set both above for real data)")

res = recalibrate_invivo(BUNDLE, ROOT, flag_distance=2.0)
plot_invivo_recalibration(res, flag=2.0)
res